# DMP Bridge — Run Pipeline

Extract and label a DMP PDF in three steps:

1. **Configure** — choose your PDF, model, and extractor below
2. **Run** — one cell runs the full pipeline
3. **Inspect** — browse the labeled blocks and structured JSON output

In [1]:
import os
from pathlib import Path

# Navigate to the project root regardless of where Jupyter started.
_cwd = Path.cwd()
if _cwd.name == "notebooks" and (_cwd.parent / "dmpbridge").exists():
    os.chdir(_cwd.parent)
elif not (_cwd / "dmpbridge").exists():
    raise RuntimeError(f"Cannot find project root from {_cwd}.")

print(f"Working directory: {Path.cwd()}")

Working directory: c:\Users\Nahid\dmpbridge


## 1 — Configuration
Edit the values below, then run all cells.

In [2]:
# ── Input ─────────────────────────────────────────────────────────────────────
PDF_PATH  = Path("data/input/pdfs/sample10.pdf")   # path to your PDF

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL     = "gemma4:e4b"              # options: "llama3.1:8b"  "llama3.3:70b"  "gemma4:e4b"
HOST      = "http://localhost:11434"  # Ollama server URL

# ── Extractor ─────────────────────────────────────────────────────────────────
EXTRACTOR = "pdfplumber"   # options: "pdfplumber"  "docling"

# ── Annotation rules ──────────────────────────────────────────────────────────
APPLY_RULES = True         # backfill empty question texts from section titles

# ── Output ────────────────────────────────────────────────────────────────────
OUT_DIR = Path("data/output/pipeline_run")
OUT_DIR.mkdir(parents=True, exist_ok=True)

stem            = PDF_PATH.stem
LABELED_JSON    = OUT_DIR / f"{stem}_labeled.json"
STRUCTURED_JSON = OUT_DIR / f"{stem}_structured.json"

print(f"PDF       : {PDF_PATH}  {'✓ exists' if PDF_PATH.exists() else '✗ NOT FOUND'}")
print(f"Model     : {MODEL}")
print(f"Extractor : {EXTRACTOR}")
print(f"Rules     : {APPLY_RULES}")
print(f"Output    : {OUT_DIR}/")

PDF       : data\input\pdfs\sample10.pdf  ✓ exists
Model     : gemma4:e4b
Extractor : pdfplumber
Rules     : True
Output    : data\output\pipeline_run/


In [3]:
import requests

try:
    r = requests.get(f"{HOST}/api/tags", timeout=5)
    r.raise_for_status()
    available = [m["name"] for m in r.json().get("models", [])]
    loaded    = any(MODEL in m for m in available)
    print(f"Ollama : running at {HOST}  ✓")
    print(f"Model  : {MODEL}  {'✓ ready' if loaded else '✗ not pulled — run: ollama pull ' + MODEL}")
except Exception:
    print(f"Ollama : NOT reachable at {HOST}  ✗")
    print("  → Start Ollama first, then re-run this cell.")
    raise SystemExit("Ollama must be running before you can run the pipeline.")

Ollama : running at http://localhost:11434  ✓
Model  : gemma4:e4b  ✓ ready


## 2 — Run the pipeline

In [4]:
import dmpbridge

blocks = dmpbridge.process_pdf(
    PDF_PATH,
    model=MODEL,
    host=HOST,
    extractor=EXTRACTOR,
    apply_rules=APPLY_RULES,
    output=LABELED_JSON,
    structured_output=STRUCTURED_JSON,
    raw_dir=None,
)

print(f"Done — {len(blocks)} blocks labeled")
print(f"Labeled JSON    → {LABELED_JSON}")
print(f"Structured JSON → {STRUCTURED_JSON}")

Done — 68 blocks labeled
Labeled JSON    → data\output\pipeline_run\sample10_labeled.json
Structured JSON → data\output\pipeline_run\sample10_structured.json


## 3 — Labeled blocks
Every text block from the PDF with its assigned label and confidence score.

In [1]:
from IPython.display import HTML
from collections import Counter

LABEL_STYLE = {
    "title":               ("#92400e", "#fef3c7", "🏷️"),
    "section.title":       ("#1e40af", "#dbeafe", "📂"),
    "section.description": ("#5b21b6", "#ede9fe", "📝"),
    "question.text":       ("#065f46", "#d1fae5", "❓"),
    "answer.text":         ("#374151", "#f3f4f6", "💬"),
}

def badge(label):
    color, bg, icon = LABEL_STYLE.get(label, ("#374151", "#f3f4f6", "•"))
    return (
        f'<span style="background:{bg};color:{color};padding:2px 8px;'
        f'border-radius:10px;font-size:11px;font-weight:600;white-space:nowrap;">'
        f'{icon} {label}</span>'
    )

rows_html = ""
for i, b in enumerate(blocks):
    label = b.get("label", "answer.text")
    conf  = b.get("confidence", 1.0)
    text  = b["text"].replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    bg    = "#ffffff" if i % 2 == 0 else "#f9fafb"
    rows_html += (
        f'<tr style="background:{bg}">'
        f'<td style="padding:7px 10px;color:#9ca3af;font-size:11px;">{b["page"]}</td>'
        f'<td style="padding:7px 10px;">{badge(label)}</td>'
        f'<td style="padding:7px 12px;font-size:13px;color:#1f2937;">{text}</td>'
        f'<td style="padding:7px 10px;font-size:11px;color:#9ca3af;text-align:right;">{conf:.0%}</td>'
        f'</tr>'
    )

counts = Counter(b.get("label", "answer.text") for b in blocks)
summary = "  ".join(
    f'<span style="margin-right:6px;">{badge(l)} <b>{n}</b></span>'
    for l, n in counts.most_common()
)

html = f"""
<div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;max-width:960px;">
  <div style="margin-bottom:12px;padding:10px 14px;background:#f8fafc;border-radius:8px;border:1px solid #e5e7eb;">
    <span style="font-size:12px;color:#6b7280;font-weight:600;margin-right:10px;">SUMMARY</span>{summary}
  </div>
  <div style="border:1px solid #e5e7eb;border-radius:8px;overflow:hidden;max-height:520px;overflow-y:auto;">
    <table style="width:100%;border-collapse:collapse;">
      <thead>
        <tr style="background:#f1f5f9;position:sticky;top:0;z-index:1;">
          <th style="padding:8px 10px;text-align:left;font-size:11px;color:#6b7280;border-bottom:1px solid #e5e7eb;">PG</th>
          <th style="padding:8px 10px;text-align:left;font-size:11px;color:#6b7280;border-bottom:1px solid #e5e7eb;">LABEL</th>
          <th style="padding:8px 12px;text-align:left;font-size:11px;color:#6b7280;border-bottom:1px solid #e5e7eb;">TEXT</th>
          <th style="padding:8px 10px;text-align:right;font-size:11px;color:#6b7280;border-bottom:1px solid #e5e7eb;">CONF</th>
        </tr>
      </thead>
      <tbody>{rows_html}</tbody>
    </table>
  </div>
</div>
"""

HTML(html)

NameError: name 'blocks' is not defined

## 4 — Structured DMP output
The final nested structure — title, sections, questions, and answers — ready for the DMP Tool.

In [6]:
import json
from IPython.display import HTML

structured = json.loads(STRUCTURED_JSON.read_text(encoding="utf-8"))
template   = structured["narrative"]["template"]

def esc(s):
    return s.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;").replace("\n", "<br>")

# ── Document title ────────────────────────────────────────────────────────────
doc_title = template.get("title", "").strip()
title_html = (
    f'<div style="background:#1e3a5f;color:white;padding:14px 18px;border-radius:8px;'
    f'margin-bottom:16px;font-size:16px;font-weight:700;">📄 {esc(doc_title)}</div>'
    if doc_title else
    '<div style="background:#e5e7eb;color:#9ca3af;padding:10px 14px;border-radius:8px;'
    'margin-bottom:16px;font-size:13px;font-style:italic;">📄 No document title detected</div>'
)

# ── Sections ──────────────────────────────────────────────────────────────────
sections_html = ""
for sec in template.get("section", []):
    sec_title = esc(sec.get("title", "").strip())
    desc      = sec.get("description", "").strip()

    desc_html = (
        f'<div style="margin:8px 0 12px 0;padding:8px 12px;background:#f5f3ff;'
        f'border-left:3px solid #7c3aed;border-radius:0 6px 6px 0;'
        f'font-size:12px;color:#5b21b6;font-style:italic;">{esc(desc)}</div>'
        if desc else ""
    )

    questions_html = ""
    for q in sec.get("question", []):
        q_text = esc(q.get("text", "").strip())
        ans    = q.get("answer", {}).get("json", {}).get("answer", "").strip()
        ans_html = (
            f'<div style="margin-top:6px;padding:8px 12px;background:#f0fdf4;'
            f'border-left:3px solid #10b981;border-radius:0 6px 6px 0;'
            f'font-size:13px;color:#374151;">{esc(ans)}</div>'
            if ans else
            '<div style="margin-top:6px;font-size:12px;color:#9ca3af;font-style:italic;">No answer</div>'
        )
        questions_html += (
            f'<div style="margin:10px 0;padding:10px 14px;background:#fafafa;'
            f'border:1px solid #e5e7eb;border-radius:6px;">'
            f'<div style="font-size:12px;font-weight:600;color:#065f46;margin-bottom:4px;">❓ Question {q["order"]}</div>'
            f'<div style="font-size:13px;color:#1f2937;font-weight:500;">{q_text}</div>'
            f'{ans_html}'
            f'</div>'
        )

    if not questions_html:
        questions_html = '<div style="font-size:12px;color:#9ca3af;font-style:italic;padding:4px 0;">No questions</div>'

    sections_html += (
        f'<div style="margin-bottom:16px;border:1px solid #bfdbfe;border-radius:8px;overflow:hidden;">'
        f'<div style="background:#1e40af;color:white;padding:10px 16px;font-weight:600;font-size:14px;">'
        f'📂 Section {sec["order"]}: {sec_title}</div>'
        f'<div style="padding:12px 16px;">{desc_html}{questions_html}</div>'
        f'</div>'
    )

n_sections  = len(template.get("section", []))
n_questions = sum(len(s.get("question", [])) for s in template.get("section", []))
stats_html  = (
    f'<div style="margin-bottom:14px;padding:8px 14px;background:#f0f9ff;border-radius:6px;'
    f'font-size:12px;color:#0369a1;">'
    f'<b>{n_sections}</b> sections &nbsp;·&nbsp; <b>{n_questions}</b> questions &nbsp;·&nbsp; '
    f'Model: <b>{MODEL}</b> &nbsp;·&nbsp; Extractor: <b>{EXTRACTOR}</b>'
    f'</div>'
)

HTML(
    f'<div style="font-family:-apple-system,BlinkMacSystemFont,\'Segoe UI\',sans-serif;max-width:900px;">'
    f'{title_html}{stats_html}{sections_html}</div>'
)